# Setup:
1. Create environment:
    In terminal, run:
    
    `conda env create -n suite2p_preprocessing_env -f studio/app/optinist/wrappers/suite2p/conda/suite2p_with_expdb.yaml`

    `conda activate suite2p_preprocessing_env`

2. Install additional packages:

   `pip install pynwb imageio ipython jupyter notebook plotly "pydantic<2.0.0" python-dotenv uvicorn xmltodict bcrypt concurrent-log-handler nd2`

3. For Windows, install correct PyTorch version (to avoid Windows DLL errors):

       `conda install -n suite2p_preprocessing_env pytorch==2.2.0 torchvision torchaudio cpuonly -c pytorch -y`
  
- If running in VS code, you may need to restart and/or select the correct environment with "Python: Select Interpreter"

4. Run this notebook

Note: This notebook demonstrates the ExpDB preprocessing workflow:
microscope_database -> preprocessing -> suite2p_preprocessing -> analyze_stats

In [ ]:
import os
import sys
import uuid
import scipy.io
import json
sys.path.append(os.path.abspath('..'))
sys.path.append(os.path.abspath('.'))

# Set environment variables for local development (non-Docker)
# Use relative paths so this works for any user on any platform (Windows/Linux/Mac)
notebook_dir = os.path.dirname(os.path.abspath('__file__')) if '__file__' in globals() else os.getcwd()
repo_root = os.path.abspath(os.path.join(notebook_dir, '..'))

# Go up one level from repo root to find experiments_datasets and experiments_public
# Use os.path.join for cross-platform compatibility (Windows/Linux/Mac)
os.environ['EXPDB_DIR'] = os.path.abspath(os.path.join(repo_root, '..', 'experiments_datasets'))
os.environ['PUBLIC_EXPDB_DIR'] = os.path.abspath(os.path.join(repo_root, '..', 'experiments_public'))

# Import OptiNiSt core data modules
from studio.app.dir_path import DIRPATH
from studio.app.common.dataclass import ImageData
from studio.app.optinist.dataclass.microscope_expdb import MicroscopeExpdbData
from studio.app.common.core.utils.filepath_creater import join_filepath
from studio.app.common.core.utils.filepath_finder import find_param_filepath
from studio.app.common.core.utils.config_handler import ConfigReader

# Import preprocessing and analysis modules
from studio.app.optinist.wrappers.expdb.preprocessing import preprocessing
from studio.app.optinist.wrappers.suite2p.suite2p_preprocessing import suite2p_preprocessing
from studio.app.optinist.wrappers.expdb.analyze_stats import analyze_stats

import numpy as np
import pandas as pd

# Import visualization modules
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import plotly.express as px

# Use EXPDB_DIR as the input directory (where experiment data is stored)
EXPDB_DIR = os.environ['EXPDB_DIR']
unique_id = str(uuid.uuid4())[:8]  # Generate 8-char unique ID

print(f"Using EXPDB_DIR: {EXPDB_DIR}")
print(f"Using PUBLIC_EXPDB_DIR: {os.environ['PUBLIC_EXPDB_DIR']}")

In [ ]:
# Step 1.1: Load microscope data from ND2 file
# Using the actual experiment data from EXPDB_DIR

# Path to the ND2 file (update this to your experiment file)
input_file = join_filepath([EXPDB_DIR, 'M000000', 'M000000_ori001', 'M000000_ori001.nd2'])

# Extract subject_id and experiment_id from the file path
# Path structure: EXPDB_DIR/subject_id/experiment_id/experiment_id.nd2
experiment_id = os.path.splitext(os.path.basename(input_file))[0]  # Get filename without extension
experiment_dir = os.path.dirname(input_file)  # Get experiment directory
subject_id = os.path.basename(os.path.dirname(experiment_dir))  # Get subject directory name

print(f"Extracted from path:")
print(f"  Subject ID: {subject_id}")
print(f"  Experiment ID: {experiment_id}")

# Verify file exists
if not os.path.exists(input_file):
    raise FileNotFoundError(f"ND2 file not found at: {input_file}")

# Create MicroscopeExpdbData wrapper for ND2 files
microscope_data = MicroscopeExpdbData(input_file)

# Get metadata from the reader
reader = microscope_data.reader
ome_meta = reader.ome_metadata

print(f"\nLoaded microscope data from: {input_file}")
print(f"Channels: {ome_meta.size_c}, Time points: {ome_meta.size_t}, Z-slices: {ome_meta.size_z}")
print(f"Image dimensions: {ome_meta.size_y} x {ome_meta.size_x}")
print(f"Expected stack shape: (ch={ome_meta.size_c}, t={ome_meta.size_t}, z={ome_meta.size_z}, y={ome_meta.size_y}, x={ome_meta.size_x})")
print(f"Imaging rate: {ome_meta.imaging_rate} Hz")

# Step 1.2: Load trial structure and metadata files
trialstructure_path = join_filepath([EXPDB_DIR, subject_id, experiment_id, f'{experiment_id}_trialstructure.mat'])
metadata_path = join_filepath([EXPDB_DIR, subject_id, experiment_id, f'{experiment_id}_metadata.json'])

# Load trialstructure.mat
trialstructure = scipy.io.loadmat(trialstructure_path)
print(f"\nLoaded trial structure from: {trialstructure_path}")

# Load metadata.json
with open(metadata_path, 'r') as f:
    metadata = json.load(f)
print(f"Loaded metadata from: {metadata_path}")

In [ ]:
# Step 2: Set parameters for preprocessing
# This step performs phase correction and optional registration

preprocessing_params = {
    'preprocessing': {
        'first_dim': 2,          # First dimension for phase correction (1 =horizontal scanning, 2=vertical)
        'period': 1,            # Period for stack averaging
        'runs': [10],              # Number of runs for averaging
        'do_realign': True,      # Enable registration/realignment
        'usfac': 10,             # Upsampling factor for registration (2D)
        'le': 6,                 # Local extrema parameter (3D)
        'shift_method': 'cv2'    # Shift method for 3D: 'cv2' or 'scipy'
    }
}

In [ ]:
# Create output directory for preprocessing
preproc_function_id = f"preprocessing_{unique_id}"
preproc_output_dir = os.path.join(DIRPATH.OUTPUT_DIR, "1", unique_id, preproc_function_id)
os.makedirs(preproc_output_dir, exist_ok=True)

In [ ]:
# Run preprocessing
print("Running preprocessing (phase correction and registration)...")

# Load NWB configuration
nwb_config = ConfigReader.read(find_param_filepath("nwb"))

# Call preprocessing with the NWB config
ret_preproc = preprocessing(
    microscope_data, 
    preproc_output_dir, 
    preprocessing_params,
    nwbfile=nwb_config
)
print(f"Preprocessing complete. Output keys: {list(ret_preproc.keys())}")

In [ ]:
# Step 3: Set parameters for suite2p_preprocessing
# This step performs ROI detection and creates ExpDB .mat files

suite2p_preprocessing_params = {
    'suite2p_preprocessing': {
        # Cell detection parameters
        'tau': 1.25,                    # Timescale of calcium indicator
        # 'fs' is auto-extracted from ND2 metadata (~2.03 Hz for this file)
        'threshold_scaling': 1.0,       # Detection threshold multiplier
        'max_overlap': 0.75,            # Maximum ROI overlap
        'spatial_hp_detect': 25,        # Spatial high-pass filter window
        'connected': True,              # Use connected components
        'high_pass': 100,               # Temporal high-pass filter
        
        # ROI extraction parameters
        'neuropil_extract': True,       # Extract neuropil traces
        'inner_neuropil_radius': 2,     # Pixels between ROI and neuropil
        'min_neuropil_pixels': 350,     # Minimum neuropil pixels
        
        # ExpDB-specific parameters
        'neucoeff': 0.7,                # Neuropil contamination coefficient
        'normalize_by_energy': True,    # Apply energy normalization (recommended)
        
        # Visualization parameters
        'roi_thr_bool': False,          # Apply energy thresholding to ROI pixels
        'roi_thr': 0.9,                 # ROI pixel energy threshold
        
        # Output control
        'create_Yr': False,             # Create Yr.mat (large file, optional)
        'create_C_or': False,           # Create C_or.mat (optional)
        'validate_outputs': True,       # Validate .mat files
        'require_trialstructure': False # Require trial structure file (set False for testing)
    }
}

In [ ]:
# Create output directory for suite2p_preprocessing
s2p_preproc_function_id = f"suite2p_preprocessing_{unique_id}"
s2p_preproc_output_dir = os.path.join(DIRPATH.OUTPUT_DIR, "1", unique_id, s2p_preproc_function_id)
os.makedirs(s2p_preproc_output_dir, exist_ok=True)

In [ ]:
# Run suite2p_preprocessing
print("Running suite2p_preprocessing (ROI detection and .mat file creation)...")
ret_s2p_preproc = suite2p_preprocessing(
    ret_preproc['stack'], 
    s2p_preproc_output_dir, 
    suite2p_preprocessing_params
)
print(f"Suite2p preprocessing complete. Output keys: {list(ret_s2p_preproc.keys())}")

In [ ]:
# Step 4: Set parameters for analyze_stats
# This step performs statistical analysis for orientation/direction tuning
# Note: All parameters should be at the top level (flattened), not nested

analyze_stats_params = {
    # stat_file_convert parameters
    'percentile_window': 30,        # Window for percentile filtering (default: 49)
    'n_percentile': 8,              # Percentile value (default: 20)
    'moving_avg_window': 3,         # Moving average window (default: 21)
    'nbinning': 1,                  # Binning for smoothing
    
    # anova1_mult parameters
    'p_value_threshold': 0.05,      # P-value threshold for significance
    'r_best_threshold': 0.05,       # Response threshold
    'si_threshold': 0.3,            # Selectivity index threshold
    
    # curvefit_tuning parameters
    'do_interpolation': True,       # Enable interpolation
    'interp_method': 'spline',      # Interpolation method
    'p_threshold': 0.05,            # P-value threshold for curve fitting
    'use_fourier': True,            # Use Fourier analysis
    
    # Stimulus parameters (for non-circular data)
    'stim_min_value': 0,            # Minimum stimulus value
    'stim_max_value': 1,            # Maximum stimulus value
    'stim_unit': 'normalized',      # Stimulus units
}

In [ ]:
# Create output directory for analyze_stats
stats_function_id = f"analyze_stats_{unique_id}"
stats_output_dir = os.path.join(DIRPATH.OUTPUT_DIR, "1", unique_id, stats_function_id)
os.makedirs(stats_output_dir, exist_ok=True)

In [ ]:
# Run analyze_stats
# Note: This requires trial structure data to be present
print("Running analyze_stats (statistical analysis)...")

# Import ExpDbData to create the proper input
from studio.app.optinist.dataclass import ExpDbData
import glob

# Get the timecourse.mat file created by suite2p_preprocessing
timecourse_files = glob.glob(os.path.join(s2p_preproc_output_dir, f"{experiment_id}_timecourse.mat"))
if not timecourse_files:
    raise FileNotFoundError(f"timecourse.mat not found in {s2p_preproc_output_dir}")
timecourse_path = timecourse_files[0]

# Use the trialstructure path from Step 1.1
# Create ExpDbData with both timecourse and trialstructure paths
expdb_data = ExpDbData(paths=[timecourse_path, trialstructure_path])

print(f"  Timecourse: {timecourse_path}")
print(f"  Trialstructure: {trialstructure_path}")

try:
    ret_stats = analyze_stats(
        expdb_data,  # Pass the ExpDbData object with both files
        stats_output_dir, 
        analyze_stats_params
    )
    print(f"Statistical analysis complete. Output keys: {list(ret_stats.keys())}")
except Exception as e:
    import traceback
    print(f"Error running analyze_stats: {e}")
    print(traceback.format_exc())
    ret_stats = None

In [ ]:
# Visualize Suite2p preprocessing results

# Get data from the output
mean_img = ret_s2p_preproc['mean_image'].data
max_proj = ret_s2p_preproc['max_proj'].data
Vcorr = ret_s2p_preproc['Vcorr'].data
cell_roi = ret_s2p_preproc['cell_roi'].data
F = ret_s2p_preproc['fluorescence'].data
iscell = ret_s2p_preproc['iscell'].data

# Create subplot figure
fig = make_subplots(
    rows=3, cols=2, 
    subplot_titles=(
        'Mean Image', 'Max Projection',
        'Correlation Image', 'Cell ROIs',
        'Mean Fluorescence', 'Individual Cell Traces'
    ),
    vertical_spacing=0.10,
    horizontal_spacing=0.12
)

# 1. Mean Image
fig.add_trace(
    go.Heatmap(
        z=mean_img, 
        colorscale='gray',
        showscale=False,
        name='Mean Image'
    ),
    row=1, col=1
)

# 2. Max Projection
fig.add_trace(
    go.Heatmap(
        z=max_proj,
        colorscale='gray',
        showscale=False,
        name='Max Projection'
    ),
    row=1, col=2
)

# 3. Correlation Image
fig.add_trace(
    go.Heatmap(
        z=Vcorr,
        colorscale='viridis',
        showscale=True,
        name='Correlation',
        colorbar=dict(
            title='Corr',
            len=0.25,
            y=0.5
        )
    ),
    row=2, col=1
)

# 4. Cell ROIs
fig.add_trace(
    go.Heatmap(
        z=cell_roi,
        colorscale='viridis',
        showscale=True,
        name='Cell ROIs',
        colorbar=dict(
            title='ROI #',
            len=0.25,
            y=0.5
        )
    ),
    row=2, col=2
)

# 5. Mean Fluorescence
mean_fluorescence = np.mean(F, axis=0)
time_points = np.arange(len(mean_fluorescence))

fig.add_trace(
    go.Scatter(
        x=time_points,
        y=mean_fluorescence,
        mode='lines',
        name='Mean Fluorescence',
        showlegend=False
    ),
    row=3, col=1
)

# 6. Individual Cell Traces
cell_indices = np.where(iscell == 1)[0]
n_cells_to_plot = min(10, len(cell_indices))
colors = px.colors.qualitative.Set3

for i in range(n_cells_to_plot):
    cell_idx = cell_indices[i]
    color = colors[i % len(colors)]
    fig.add_trace(
        go.Scatter(
            x=time_points,
            y=F[cell_idx, :],
            mode='lines',
            name=f'Cell {cell_idx+1}',
            line=dict(color=color, width=1),
            opacity=0.7,
            showlegend=(i < 5)  # Only show first 5 in legend
        ),
        row=3, col=2
    )

# Update layout
fig.update_layout(
    height=900,
    width=1200,
    title=dict(
        text=f"Suite2p Preprocessing Results ({np.sum(iscell)} cells detected)",
        x=0.5,
        y=0.98
    ),
    showlegend=True
)

# Update axes
for row in range(1, 4):
    for col in range(1, 3):
        if row <= 2:  # Images
            fig.update_xaxes(title_text="X", row=row, col=col)
            fig.update_yaxes(title_text="Y", row=row, col=col)
        else:  # Traces
            fig.update_xaxes(title_text="Time (frames)", row=row, col=col)
            fig.update_yaxes(title_text="Fluorescence", row=row, col=col)

fig.show()

In [ ]:
# Visualize analyze_stats results (if available)
# Using the same plotting functions as the batch processing system

if ret_stats is not None:
    # Get statistical analysis outputs
    stat_data = ret_stats['stat']
    
    # Create output directory for plots
    plot_output_dir = os.path.join(stats_output_dir, 'plots')
    os.makedirs(plot_output_dir, exist_ok=True)
    
    # Check if data is circular (orientation/direction) based on experiment ID
    # Circular patterns: '_ori', '_OF-PRC-', '_dot'
    is_circular = any(pattern in experiment_id for pattern in ['_ori', '_OF-PRC-', '_dot'])
    print(f"Data type: {'Circular (orientation/direction)' if is_circular else 'Non-circular (stimulus)'}")
    print(f"\nGenerating plots using system plotting functions...")
    
    if is_circular:
        # Generate circular data plots (orientation/direction)
        print("Generating orientation/direction plots:")
        
        # Tuning curves
        stat_data.tuning_curve.save_plot(plot_output_dir)
        print("  ✓ tuning_curve.png")
        
        stat_data.tuning_curve_polar.save_plot(plot_output_dir)
        print("  ✓ tuning_curve_polar.png")
        
        # Responsivity ratios (pie charts)
        stat_data.direction_responsivity_ratio.save_plot(plot_output_dir)
        print("  ✓ responsivity_ratio.png (direction)")
        
        stat_data.orientation_responsivity_ratio.save_plot(plot_output_dir)
        print("  ✓ orientation_responsivity_ratio.png")
        
        # Selectivity indices (histograms)
        stat_data.direction_selectivity.save_plot(plot_output_dir)
        print("  ✓ selectivity.png (direction)")
        
        stat_data.orientation_selectivity.save_plot(plot_output_dir)
        print("  ✓ orientation_selectivity.png")
        
        # Best responsivity
        stat_data.best_responsivity.save_plot(plot_output_dir)
        print("  ✓ best_responsivity.png")
        
        # Preferred angles
        stat_data.preferred_direction.save_plot(plot_output_dir)
        print("  ✓ preferred_direction.png")
        
        stat_data.preferred_orientation.save_plot(plot_output_dir)
        print("  ✓ preferred_orientation.png")
        
        # Tuning widths
        stat_data.direction_tuning_width.save_plot(plot_output_dir)
        print("  ✓ direction_tuning_width.png")
        
        stat_data.orientation_tuning_width.save_plot(plot_output_dir)
        print("  ✓ orientation_tuning_width.png")
    else:
        # Generate non-circular data plots (stimulus)
        print("Generating stimulus response plots:")
        
        stat_data.stim_tuning_curve.save_plot(plot_output_dir)
        print("  ✓ stim_tuning_curve.png")
        
        stat_data.stim_selectivity.save_plot(plot_output_dir)
        print("  ✓ stim_selectivity.png")
        
        stat_data.stim_responsivity.save_plot(plot_output_dir)
        print("  ✓ stim_responsivity.png")
        
        stat_data.stim_responsivity_ratio.save_plot(plot_output_dir)
        print("  ✓ stim_responsivity_ratio.png")
    
    print(f"\nPlots saved to: {plot_output_dir}")
    
    # Display key plots inline
    from IPython.display import Image, display
    import glob
    
    print("\n" + "="*60)
    print("DISPLAYING KEY PLOTS")
    print("="*60)
    
    # Display the first tuning curve
    tuning_curve_files = sorted(glob.glob(os.path.join(plot_output_dir, "tuning_curve_*.png")))
    tuning_curve_files = [f for f in tuning_curve_files if '.thumb.' not in f]
    
    if tuning_curve_files:
        first_tuning_curve = tuning_curve_files[0]
        plot_name = os.path.basename(first_tuning_curve)
        print(f"\n{plot_name}:")
        display(Image(filename=first_tuning_curve, width=600))
        print(f"Note: {len(tuning_curve_files)} total tuning curve plots generated")
    else:
        print("No tuning curve plots found")
    
    # Display best_responsivity
    best_resp_path = os.path.join(plot_output_dir, "best_responsivity.png")
    if os.path.exists(best_resp_path):
        print(f"\nbest_responsivity.png:")
        display(Image(filename=best_resp_path, width=600))
    
    # Display preferred_direction
    pref_dir_path = os.path.join(plot_output_dir, "preferred_direction.png")
    if os.path.exists(pref_dir_path):
        print(f"\npreferred_direction.png:")
        display(Image(filename=pref_dir_path, width=600))
    
    # Display preferred_orientation
    pref_ori_path = os.path.join(plot_output_dir, "preferred_orientation.png")
    if os.path.exists(pref_ori_path):
        print(f"\npreferred_orientation.png:")
        display(Image(filename=pref_ori_path, width=600))
    
    # Print summary statistics
    print("\n" + "="*60)
    print("STATISTICAL ANALYSIS SUMMARY")
    print("="*60)
    print(f"Total cells analyzed: {stat_data.ncells}")
    
    if is_circular:
        print(f"Visually responsive cells: {stat_data.ncells_visually_responsive_cell}")
        print(f"Direction-selective cells: {stat_data.ncells_direction_selective_cell}")
        print(f"Orientation-selective cells: {stat_data.ncells_orientation_selective_cell}")
        
        # Get direction and orientation selectivity data
        dir_sel = ret_stats['direction_selectivity'].data
        ori_sel = ret_stats['orientation_selectivity'].data
        dir_sel = np.asarray(dir_sel).flatten()
        ori_sel = np.asarray(ori_sel).flatten()
        
        print(f"\nMean Direction Selectivity Index: {np.mean(dir_sel):.3f} ± {np.std(dir_sel):.3f}")
        print(f"Mean Orientation Selectivity Index: {np.mean(ori_sel):.3f} ± {np.std(ori_sel):.3f}")
    else:
        print(f"Stimulus responsive cells: {stat_data.ncells_stim_responsive_cell}")
        print(f"Stimulus selective cells: {stat_data.ncells_stim_selective_cell}")
    
else:
    print("Statistical analysis results not available")
    print("Requirements:")
    print("1. Trial structure .mat file in the ExpDB directory")
    print("2. Proper parameter configuration")

In [ ]:
# Summary of the workflow

print("\n=== Workflow Summary ===")
print("This notebook demonstrates the complete ExpDB preprocessing pipeline:\n")
print("1. microscope_database (ND2Reader or TIFF)")
print("   └─> Loads microscope image data\n")
print("2. preprocessing")
print("   └─> Phase correction, averaging, and registration")
print(f"   └─> Output: {ret_preproc['stack'].data.shape} image stack\n")
print("3. suite2p_preprocessing")
print("   └─> Suite2p ROI detection")
print("   └─> Fluorescence extraction with neuropil correction")
print("   └─> Creates ExpDB-compatible .mat files (timecourse.mat)")
print(f"   └─> Detected: {np.sum(iscell)} cells\n")
if ret_stats is not None:
    print("4. analyze_stats")
    print("   └─> Statistical analysis (ANOVA, tuning curves)")
    print("   └─> Direction and orientation selectivity metrics")
    print(f"   └─> Analyzed: {ret_stats['stat'].ncells} cells")
else:
    print("4. analyze_stats")
    print("   └─> Not run (requires trial structure file)")
    print("   └─> See cell above for requirements")